In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
print("All packages ready!")

All packages ready!


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib

df = pd.read_csv('cybersecurity_attacks.csv')
print(df.shape)
print(df.columns.tolist())

(40000, 25)
['Timestamp', 'Source IP Address', 'Destination IP Address', 'Source Port', 'Destination Port', 'Protocol', 'Packet Length', 'Packet Type', 'Traffic Type', 'Payload Data', 'Malware Indicators', 'Anomaly Scores', 'Alerts/Warnings', 'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level', 'User Information', 'Device Information', 'Network Segment', 'Geo-location Data', 'Proxy Information', 'Firewall Logs', 'IDS/IPS Alerts', 'Log Source']


Features shape: (40000, 7)
Missing values: 40000
   Severity Level  Attack Type Enc  Anomaly Scores  Packet Length  \
0             NaN                2           28.67            503   
1             NaN                2           51.50           1174   
2             NaN                0           87.42            306   
3             NaN                2           15.79            385   
4             NaN                0            0.52           1462   

   Protocol Enc  Geo Enc  Segment Enc  
0             0     3723            0  
1             0     1677            1  
2             2     1707            2  
3             2     3750            1  
4             1      577            2  


In [6]:
# Check exact column names and severity values
print(df.columns.tolist())
print(df['Severity Level'].unique()[:10])
print(df['Severity Level'].dtype)

['Timestamp', 'Source IP Address', 'Destination IP Address', 'Source Port', 'Destination Port', 'Protocol', 'Packet Length', 'Packet Type', 'Traffic Type', 'Payload Data', 'Malware Indicators', 'Anomaly Scores', 'Alerts/Warnings', 'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level', 'User Information', 'Device Information', 'Network Segment', 'Geo-location Data', 'Proxy Information', 'Firewall Logs', 'IDS/IPS Alerts', 'Log Source', 'Attack Type Enc', 'Protocol Enc', 'Geo Enc', 'Segment Enc']
[nan]
float64


In [7]:
from sklearn.preprocessing import LabelEncoder

# Severity Level is empty in this dataset — drop it, use Anomaly Scores instead

# Fill missing NUMBER columns
df['Anomaly Scores'] = pd.to_numeric(df['Anomaly Scores'], errors='coerce')
df['Anomaly Scores'] = df.groupby('Attack Type')['Anomaly Scores'].transform(
    lambda x: x.fillna(x.median())
)

df['Packet Length'] = pd.to_numeric(df['Packet Length'], errors='coerce')
df['Packet Length'] = df.groupby('Protocol')['Packet Length'].transform(
    lambda x: x.fillna(x.median())
)

# Fill missing TEXT columns with mode
df['Attack Type'] = df['Attack Type'].fillna(df['Attack Type'].mode()[0])
df['Protocol'] = df['Protocol'].fillna(df['Protocol'].mode()[0])
df['Geo-location Data'] = df['Geo-location Data'].fillna(df['Geo-location Data'].mode()[0])
df['Network Segment'] = df['Network Segment'].fillna(df['Network Segment'].mode()[0])

# Encode text → numbers
le_attack = LabelEncoder()
le_proto = LabelEncoder()
le_geo = LabelEncoder()
le_seg = LabelEncoder()

df['Attack Type Enc'] = le_attack.fit_transform(df['Attack Type'])
df['Protocol Enc'] = le_proto.fit_transform(df['Protocol'])
df['Geo Enc'] = le_geo.fit_transform(df['Geo-location Data'])
df['Segment Enc'] = le_seg.fit_transform(df['Network Segment'])

# Final features (no Severity Level — it's empty)
features = df[['Attack Type Enc', 'Anomaly Scores',
               'Packet Length', 'Protocol Enc', 'Geo Enc', 'Segment Enc']].copy()

print("Features shape:", features.shape)
print("Missing values:", features.isnull().sum().sum())
print(features.head())

Features shape: (40000, 6)
Missing values: 0
   Attack Type Enc  Anomaly Scores  Packet Length  Protocol Enc  Geo Enc  \
0                2           28.67            503             0     3723   
1                2           51.50           1174             0     1677   
2                0           87.42            306             2     1707   
3                2           15.79            385             2     3750   
4                0            0.52           1462             1      577   

   Segment Enc  
0            0  
1            1  
2            2  
3            1  
4            2  


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(features, test_size=0.2, random_state=42)

model = IsolationForest(contamination=0.1, random_state=42)
model.fit(X_train)

predictions = model.predict(X_test)
print(f"Total test rows: {len(X_test)}")
print(f"Anomalies detected: {(predictions == -1).sum()}")
print(f"Normal detected: {(predictions == 1).sum()}")

Total test rows: 8000
Anomalies detected: 744
Normal detected: 7256


In [9]:
joblib.dump(model, 'threat_model.pkl')
joblib.dump(le_attack, 'le_attack.pkl')
joblib.dump(le_proto, 'le_proto.pkl')
joblib.dump(le_geo, 'le_geo.pkl')
joblib.dump(le_seg, 'le_seg.pkl')
print("All saved!")

All saved!


In [10]:
import os
print(os.getcwd())

C:\Users\hp\Desktop\University\4th Semester\Projects\VP\ThreatCastPK\ThreatCastPK.ML
